# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [87]:
# import the requests library (1 line)
import requests

# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com/"

# assign the /status endpoint to another variable called status_url (1 line)
status_url = "https://country-leaders.onrender.com/status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req_variable = requests.get(status_url)

# check the status_code using a condition and print appropriate messages (4 lines)
if req_variable.status_code == 200:
    print(f"It works! {req_variable}")
else:
    print(f"error {req_variable}")

It works! <Response [200]>


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [88]:
# Set the countries_url variable (1 line)
countries_url = "https://country-leaders.onrender.com/countries"
# query the /countries endpoint using the get() method and store it in the req variable (1 line)
req_count = requests.get(countries_url)
# Get the JSON content and store it in the countries variable (1 line)
json_content = req_count.json()
# display the request's status code and the countries variable (1 line)
print(f"{req_count.status_code} + {json_content}")

403 + {'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [148]:
# Set the cookie_url variable (1 line)
cookie_url = "https://country-leaders.onrender.com/cookie"

# Query the enpoint, set the cookies variable and display it (2 lines)
cookies =requests.get(cookie_url).cookies
print(cookies)

<RequestsCookieJar[<Cookie user_cookie=6d89bc43-b2c3-44cf-99bb-68c16942b7a4 for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [149]:
# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(countries_url , cookies = cookies).json()
# display the countries variable (1 line)
print(countries)

['fr', 'us', 'be', 'ma', 'ru']


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [ ]:
# Set the leaders_url variable (1 line)
leaders_url = "https://country-leaders.onrender.com/    "

# query the /leaders endpoint, assign the output to the leaders variable (1 line)


# display the leaders variable (1 line)


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [ ]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
params = {
    "country" : 'fr'
}
# assign the output to the leaders variable (1 line)    
leaders = requests.get(leaders_url, params, cookies = cookies).json()
# display the leaders variable (1 line)
print(leaders)

KeyboardInterrupt: 

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [ ]:
# 4 lines
leaders_per_country ={}
for c in countries: 
    params = {
    "country" : ''
    }
    params.update({"country" : c})
    leaders = requests.get(leaders_url, params, cookies = cookies).json()
    leaders_per_country.update({ c : leaders})
    print({ c : leaders})
print(leaders_per_country) 

{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

In [ ]:
# or 1 line


It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [93]:
# < 15 lines
def get_leaders():
    countries_url = "https://country-leaders.onrender.com/countries"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    leaders_url = "https://country-leaders.onrender.com/leaders"
    
    params = {
    "country" : ''
    }
    
    cookies =requests.get(cookie_url).cookies
    countries = requests.get(countries_url , cookies = cookies).json()
   
    leaders_per_country ={}
    
    for c in countries: 
        params.update({"country" : c})
        leaders = requests.get(leaders_url, params, cookies = cookies).json()
        leaders_per_country.update({ c : leaders})
    
    return leaders_per_country 


Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [ ]:
# 2 lines
leaders_per_country = get_leaders()

print(leaders_per_country)


{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [ ]:
# 3 lines
leaders = requests.get(leaders_url, params, cookies = cookies).json()
headers = {
    "User-Agent": "Mozilla/5.0"
}
#for l in leaders:
    #print(requests.get(l['wikipedia_url'], headers=headers).text)
leader_html = requests.get(leaders[0]['wikipedia_url'], headers=headers).text




Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [ ]:
# 3 lines
from bs4 import BeautifulSoup
soup = BeautifulSoup(leader_html, 'html.parser')

print(soup.prettify())

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="ru">
 <head>
  <meta charset="utf-8"/>
  <title>
   Путин, Владимир Владимирович — Википедия
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [ ]:
# 2 lines
paragraphs = soup.find_all("p")
for p in paragraphs:
    text = p.get_text()
    if len(text)> 30:
        print(text)
        break
        




Влади́мир Влади́мирович Пу́тин (род. 7 октября 1952, Ленинград, СССР) — российский государственный и политический деятель. Действующий президент Российской Федерации, председатель Государственного Совета Российской Федерации и Совета Безопасности Российской Федерации; Верховный главнокомандующий Вооружёнными силами Российской Федерации с 7 мая 2012 года. Ранее занимал должность президента с 7 мая 2000 по 7 мая 2008 года, также в 1999—2000 и 2008—2012 годах занимал должность председателя правительства Российской Федерации. Фактически руководит Россией, по разным оценкам, с 1999[6] или с 2000 года[7]. Основатель собственной политической идеологии и движения — «путинизма»[8].


If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [ ]:
# <10 lines

leader_name = leaders[0]['first_name'] +" "+ leaders[0]['last_name']
paragraphs = soup.find_all("p")
for p in paragraphs:
    if p.text.startswith(leader_name):
        print(p.text)
        break


At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [ ]:

# 10 lines
# def get_first_paragraph(wikipedia_url):
def get_first_paragraph(wikipedia_url):
    leaders_url = "https://country-leaders.onrender.com/leaders"
    cookie_url = "https://country-leaders.onrender.com/cookie"

    # Query the enpoint, set the cookies variable and display it (2 lines)
    cookies =requests.get(cookie_url).cookies
    
#print(wikipedia_url) # keep this for the rest of the notebook
    print(wikipedia_url)
#[insert your code]
    headers = {
        "User-Agent": "wikibecode (victorcourtois135@gmail.com)"
    }
    
    leader_html = requests.get(wikipedia_url, headers=headers).text
   
    soup = BeautifulSoup(leader_html, 'html.parser')
    paragraphs = soup.find_all("p")
    leaders = requests.get(leaders_url, params, cookies = cookies).json()
    leader_name = leaders[0]['first_name']
    
    first_paragraph=""
    for p in paragraphs:
        text = p.get_text()
        if len(text)>150 :
            print(text)
            first_paragraph = text
            break

#   return first_paragraph
    return first_paragraph

In [ ]:
# Test: 3 lines
import time

for countries, presidents in leaders_per_country.items():
    for president in presidents:
        time.sleep(2)
        url = president["wikipedia_url"]
        get_first_paragraph(url)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
Vladimir
François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.

https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
Vladimir
Nicolas Sarközy de Nagy-Bocsa, dit Nicolas Sarkozy (/ni.kɔ.la saʁ.kɔ.zi/[d] Écouterⓘ ; en hongrois Sárközy ou Sárközi [ˈʃaːɾkøzi][3],[4],[5]), né le 28 janvier 1955 à Paris 17e (Seine), est un homme d'État français. Il est président de la République française du 16 mai 2007 au 15 mai 2012.

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
Vladimir
François Mitterrand ([fʁɑ̃swa mitɛʁɑ̃][b] Écouterⓘ), né le 26 octobre 1916 à Jarnac (Charente)[1] et mort le 8 janvier 1996 dans le 7e arrondissement de Paris, est un homme d'État français. Il est président de la République française du 21 mai 1981 au 17 mai 1995.

https://fr.wikipedia.org/wiki/Charles

KeyboardInterrupt: 

### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [ ]:
# 3 lines
import re
def clean_text(text):
    text_clean = re.sub(r"\(.*?\)", "", text)
    text_clean = re.sub(r"\[.*?\]", "", text_clean)
    text_clean = re.sub(r"[^a-zA-Z0-9àâäéèêëîïôöùûüçÀÂÄÉÈÊËÎÏÔÖÙÛÜÇ\s']", "", text_clean)
    text_clean = re.sub(r"\s+", " ", text_clean)
    
    return text_clean



Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [ ]:
# 10 lines
for countries, presidents in leaders_per_country.items():
    for president in presidents:
        time.sleep(2)
        url = president["wikipedia_url"]
        print(clean_text(get_first_paragraph(url)))

François Hollande  Écouter né le 12 août 1954 à Rouen  est un haut fonctionnaire et homme dÉtat français Il est président de la République française du 15 mai 2012 au 14 mai 2017



KeyboardInterrupt: 

Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [ ]:
# < 20 lines

def get_first_paragraph(wikipedia_url, session):
    
#print(wikipedia_url) # keep this for the rest of the notebook
    print(wikipedia_url)
#[insert your code]
    headers = {
        "User-Agent": "wikibecode (victorcourtois135@gmail.com)"
    }
    
    leader_html = session.get(wikipedia_url, headers=headers).text
   
    soup = BeautifulSoup(leader_html, 'html.parser')
    paragraphs = soup.find_all("p") 
    first_paragraph=""
    for p in paragraphs:
        text = p.get_text()
        if len(text)>150 :
            first_paragraph = text
            break
        
    clean_para = clean_text(first_paragraph)
    return clean_para

for countries, presidents in leaders_per_country.items():
    for president in presidents:
        time.sleep(2)
        url = president["wikipedia_url"]
        print(get_first_paragraph(url))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande Écouter né le 12 août 1954 à Rouen est un haut fonctionnaire et homme d'État français Il est président de la République française du 15 mai 2012 au 14 mai 2017 
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
Nicolas Sarközy de NagyBocsa dit Nicolas Sarkozy né le 28 janvier 1955 à Paris 17e est un homme d'État français Il est président de la République française du 16 mai 2007 au 15 mai 2012 


KeyboardInterrupt: 

## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
# < 20 lines
def get_leaders():
    countries_url = "https://country-leaders.onrender.com/countries"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    leaders_url = "https://country-leaders.onrender.com/leaders"
    
    params = {
    "country" : ''
    }
    
    cookies =requests.get(cookie_url).cookies
    countries = requests.get(countries_url , cookies = cookies).json()
   
    leaders_per_country ={}
    
    for c in countries: 
        params.update({"country" : c})
        leaders = requests.get(leaders_url, params, cookies = cookies).json()
        leaders_per_country.update({ c : leaders})
        
    
    for countries, presidents in leaders_per_country.items():
        for president in presidents:
            time.sleep(2)
            url = president["wikipedia_url"]
            president['first_paragraph'] = get_first_paragraph(url)

    
    return leaders_per_country 


In [117]:
# Check the output of your function (2 lines)
leaders_per_country = get_leaders()

print(leaders_per_country)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
# < 25 lines



Check the output of your function again.

In [ ]:
# Check the output of your function (1 line)


Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [122]:
# < 20 lines
def get_first_paragraph(wikipedia_url, session):
    
#print(wikipedia_url) # keep this for the rest of the notebook
    print(wikipedia_url)
#[insert your code]
    headers = {
        "User-Agent": "wikibecode (victorcourtois135@gmail.com)"
    }
    
    leader_html = session.get(wikipedia_url, headers=headers).text
   
    soup = BeautifulSoup(leader_html, 'html.parser')
    paragraphs = soup.find_all("p") 
    first_paragraph=""
    for p in paragraphs:
        text = p.get_text()
        if len(text)>150 :
            first_paragraph = text
            break
        
    clean_para = clean_text(first_paragraph)
    return clean_para

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [131]:
# <25 lines
class Session():
    def __init__(self):
        session = requests.Session()
        session.headers.update({"User-Agent": "wikibecode (victorcourtois135@gmail.com)"})
        self.session = session
    
    
def get_leaders(session):
    countries_url = "https://country-leaders.onrender.com/countries"
    cookie_url = "https://country-leaders.onrender.com/cookie"
    leaders_url = "https://country-leaders.onrender.com/leaders"
    
    params = {
    "country" : ''
    }
    
    cookies =requests.get(cookie_url).cookies
    countries = requests.get(countries_url , cookies = cookies).json()
   
    leaders_per_country ={}
    
    for c in countries: 
        params.update({"country" : c})
        leaders = requests.get(leaders_url, params, cookies = cookies).json()
        leaders_per_country.update({ c : leaders})
        
    
    for countries, presidents in leaders_per_country.items():
        for president in presidents:
            url = president["wikipedia_url"]
            president['first_paragraph'] = get_first_paragraph(url,session)

    
    return leaders_per_country 


Test your new functions.



In [132]:
create_session = Session()
print(get_leaders(create_session.session))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac


KeyboardInterrupt: 

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [134]:
# 3 lines
import json 
leaders_per_country = get_leaders(create_session.session)
with open('leaders.json','w') as leader_file:
    json.dump(leaders_per_country, leader_file)
    


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [135]:
# 3 lines
with open('leaders.json', "r") as leader_file:
    output = leader_file.read()
    print(output)
    


{"fr": [{"id": "Q157", "first_name": "Fran\u00e7ois", "last_name": "Hollande", "birth_date": "1954-08-12", "death_date": null, "place_of_birth": "Rouen", "wikipedia_url": "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande", "start_mandate": "2012-05-15", "end_mandate": "2017-05-14", "first_paragraph": "Fran\u00e7ois Hollande \u00c9couter n\u00e9 le 12 ao\u00fbt 1954 \u00e0 Rouen est un haut fonctionnaire et homme d'\u00c9tat fran\u00e7ais Il est pr\u00e9sident de la R\u00e9publique fran\u00e7aise du 15 mai 2012 au 14 mai 2017 "}, {"id": "Q329", "first_name": "Nicolas", "last_name": "Sarkozy", "birth_date": "1955-01-28", "death_date": null, "place_of_birth": "Paris", "wikipedia_url": "https://fr.wikipedia.org/wiki/Nicolas_Sarkozy", "start_mandate": "2007-05-16", "end_mandate": "2012-05-15", "first_paragraph": "Nicolas Sark\u00f6zy de NagyBocsa dit Nicolas Sarkozy n\u00e9 le 28 janvier 1955 \u00e0 Paris 17e est un homme d'\u00c9tat fran\u00e7ais Il est pr\u00e9sident de la R\u00e9publ

Make a function `save(leaders_per_country)` to call this code easily.

In [145]:
# 3 lines
import json
def save(leaders_per_country):
    with open('leaders.json','w') as leader_file:
        json.dump(leaders_per_country, leader_file)

In [146]:
# Call the function (1 line)
leaders_per_country = get_leaders(create_session.session)
saving = save(leaders_per_country)


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers


KeyboardInterrupt: 

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!